# 🧪 综合实验 1：~~人工智能基础与~~生成式大语言模型思维链推理

本实验是**模型推理**章节的综合实践，整合了前期课程中学习的所有关键技能：模型加载、API 调用、文本生成、以及模型评测。

## 🎯 实验目标

1. **挑战一：模型作答** — 使用本地 LLM（Qwen2-0.5B）和云端推理模型（API）分别完成 60 道数学和逻辑推理题，直观感受不同模型在推理任务上的能力差异。
2. **挑战二：模型评估** — 运用 Exact Match、F1 Score、BLEU Score 等多种评测指标，对模型的作答结果进行科学、量化的评估，并可视化对比分析。

## 📊 数据集概览

| 类型 | 难度 | 题数 |
|------|------|------|
| 数学 (math) | easy / medium / hard | 各 10 题 |
| 逻辑 (logic) | easy / medium / hard | 各 10 题 |
| **合计** | | **60 题** |

---
## 🛠️ 1. 环境初始化

### 导入依赖库

In [ ]:
import os
import json
import time
import re
import torch
import warnings
import transformers
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from openai import OpenAI

# 屏蔽不影响功能的警告信息
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=UserWarning)

# 设置中文显示环境
target_fonts = ['SimHei', 'Arial Unicode MS', 'Microsoft YaHei', 'DejaVu Sans']
available_fonts = [f.name for f in matplotlib.font_manager.fontManager.ttflist]
for font in target_fonts:
    if font in available_fonts:
        plt.rcParams['font.sans-serif'] = [font]
        break
plt.rcParams['axes.unicode_minus'] = False

print("✅ 所有依赖库导入成功！")

### 加载本地模型（Qwen2-0.5B-Instruct）

我们加载在前面实验中已经非常熟悉的 `Qwen2-0.5B-Instruct` 作为**本地基础大语言模型 (LLM)**。
它将作为我们的"基本选手"，与云端强力推理模型进行正面对决。

In [ ]:
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

def load_model(repo_id="Qwen/Qwen2-0.5B-Instruct"):
    """加载本地 Qwen2 模型"""
    model_name = repo_id.split("/")[-1]
    local_path = Path("models") / model_name
    
    print(f"🔍 正在校验模型文件：{local_path}...")
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(local_path),
        local_dir_use_symlinks=False,
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"]
    )
    
    tokenizer = AutoTokenizer.from_pretrained(str(local_path))
    model = AutoModelForCausalLM.from_pretrained(
        str(local_path),
        device_map="cpu",
        torch_dtype="auto"
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    return model, tokenizer

model, tokenizer = load_model()
print("✅ 本地模型 Qwen2-0.5B-Instruct 加载成功！")

### 配置云端 API

我们将通过 OpenAI 兼容接口调用一个**更强大的云端模型**作为推理模型（Reasoning Model）。
在第 13 课中我们已经学习了如何使用 System Prompt 和 API 调用，这里将直接复用该技能。

In [ ]:
# ============================================================
# 👇 把你的 API Key 填在下面的引号里
# ============================================================
API_KEY = ""
API_PROVIDER = "DashScope"  # 可选: "DashScope" 或 "SiliconFlow"

if API_PROVIDER == "DashScope":
    BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
    API_MODEL = "qwen3.5-flash"  # 云端推理模型
else:
    BASE_URL = "https://api.siliconflow.cn/v1"
    API_MODEL = "deepseek-ai/DeepSeek-V3.2"

api_client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print(f"✅ API 客户端已配置：{API_PROVIDER} / {API_MODEL}")

### 定义生成工具函数

我们统一定义两个核心的对话生成函数：
- `chat_with_local()`：使用本地 Qwen2 生成 response
- `chat_with_api()`：使用云端 API 生成 response

两个函数有**相同的输入/输出签名**，这样后续可以用统一的框架调用。

In [ ]:
def chat_with_local(user_prompt, system_prompt="你是一个严谨的助手。", max_new_tokens=512):
    """使用本地模型生成 response"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.pad_token_id
        )
        response_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        return tokenizer.decode(response_ids, skip_special_tokens=True).strip()


def chat_with_api(user_prompt, system_prompt="你是一个严谨的助手。", max_new_tokens=512):
    """使用云端 API 模型生成 response"""
    try:
        response = api_client.chat.completions.create(
            model=API_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=max_new_tokens,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"[API 调用失败: {e}]"


# ==========================================================
# 🚀 快速验证 (仅仅确保基础函数工作正常)
# ==========================================================
test_system_prompt = "请你扮演一个严谨的逻辑与数学教授，提供清晰的思路。"
user_prompt_math = "一个农夫有一批马，第一次卖掉总数的一半多半匹，第二次卖掉剩下的一半多半匹，第三次又卖掉剩下的一半多半匹，这时发现马全卖光了。请问农夫一开始有多少匹马？"
user_prompt_logic = "岛上有A和B两人，骑士只说真话，骗子只说假话。A说：‘我们两人中至少有一个是骗子’。请问A和B的身份分别是什么？"

print("🧪 本地模型测试 (数学):\n", chat_with_local(user_prompt_math, system_prompt=test_system_prompt))
print("\n" + "="*40 + "\n")
print("🧪 API 模型测试 (逻辑):\n", chat_with_api(user_prompt_logic, system_prompt=test_system_prompt))

---
## 📚 2. 数据载入与预览

In [ ]:
# 载入测试数据集
with open("data/test1.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"📦 数据集载入成功，共 {len(test_data)} 道题目")

# 转换为 DataFrame 方便预览
df_test = pd.DataFrame(test_data)
print("\n📊 数据分布统计：")
print(df_test.groupby(["category", "difficulty"]).size().unstack(fill_value=0))
print("\n📝 前5题预览：")
df_test.head()

---
## 🚀 挑战一：模型推理对战

在这个挑战中，我们将让两个模型分别作答 60 道题目：
- **选手 A**：本地 Qwen2-0.5B-Instruct（基础大语言模型）
- **选手 B**：云端 API 模型（推理增强型模型）

### 核心设计
- 统一的response 生成函数，确保两个模型在完全相同的条件下作答
- 实时进度追踪，不让你干等
- 记录每道题的耗时，量化「推理成本」

### 答案格式控制与提取工具

为了规范输出，我们将提示模型把最终答案放入 `<answer></answer>` 标签中。

在最终输出提取阶段，我们会先用验证函数 `check_answer_tags` 快速检测：**是否存在且仅存在一对合法标签**。如果格式不合法，直接判定为「未检测到答案」，以避免不可控的死循环和多余 Token 消耗。

In [ ]:
def check_answer_tags(text):
    """快速检测文本中是否存在且仅存在一对 <answer></answer> 标签"""
    if not text:
        return False
    return text.count("<answer>") == 1 and text.count("</answer>") == 1

def extract_final_answer(raw_response):
    """提取 <answer></answer> 标签内的内容"""
    if not raw_response or raw_response.startswith("[API 调用失败"):
        return raw_response
        
    # 标签检测
    if not check_answer_tags(raw_response):
        return "未检测到答案"
    
    # 提取标签内的内容
    match = re.search(r'<answer>(.*?)</answer>', raw_response, re.DOTALL)
    if match:
        return match.group(1).strip()
        
    return "未检测到答案"

# 验证提取工具
test_responses = [
    "经过计算，答案是 <answer>579</answer>。",
    "分析过程如下：首先...最终结果为 <answer> 42 </answer>。",
    "这个答案无法计算。",
    "有两个答案：<answer>1</answer> 和 <answer>2</answer>"
]
for resp in test_responses:
    print(f"  原文: {resp[:40]}... → 提取: {extract_final_answer(resp)}")

### Response 生成框架

这是整个实验的**核心函数**。它接收题目列表和生成函数，批量让模型作答，并记录：
- 模型的完整 response（`raw_response`）
- 提取出的最终答案（`extracted_answer`）
- 每道题的耗时（`latency`）

> **💡 学生任务**：请根据题目类型设计合适的系统提示词（System Prompt），引导模型以更高质量完成推理任务。

In [ ]:
def generate_results(questions, chat_fn, model_name, system_prompt):
    """批量生成模型验证结果
    
    Args:
        questions: 题目列表 (来自 test1.json)
        chat_fn:   生成函数 (chat_with_local 或 chat_with_api)
        model_name: 模型名称标识
        system_prompt: 测试使用的系统提示词
    
    Returns:
        results: 测试结果列表
    """
    results = []
    total = len(questions)
    
    print(f"\n{'='*60}")
    print(f"🤖 模型 [{model_name}] 开始作答，共 {total} 题")
    print(f"{'='*60}")
    
    pbar = tqdm(questions, desc=f"{model_name} 作答中", unit="题")
    
    for item in pbar:
        qid = item["id"]
        question = item["question"]
        category = item["category"]
        difficulty = item["difficulty"]
        
        pbar.set_postfix_str(f"第{qid}题 [{category}/{difficulty}]")
        
        user_prompt = f"{question}"
        
        start_time = time.time()
        raw_response = chat_fn(user_prompt, system_prompt=system_prompt)
        latency = time.time() - start_time
        
        # =======================================================
        # TODO: 自定义 Thinking 模块 (学生任务)
        # 学生可在此处编写逻辑，将思考过程单独抽离到 thinking 变量中。
        # 例如匹配特定标签符或分割文本。
        # =======================================================
        thinking = raw_response
        
        # =======================================================
        # TODO: 补全response=f"",让extract_final_answer能够正确
        # 抓取答案。
        # =======================================================
        response = f""
        final_response = chat_fn(response, system_prompt="")
        
        # 提取最终答案 (我们将依赖规范过的 final_response)
        extracted = extract_final_answer(final_response)
        
        results.append({
            "id": qid,
            "category": category,
            "difficulty": difficulty,
            "question": question,
            "expected_answer": item["expected_answer"],
            "raw_response": raw_response,
            "thinking": thinking,
            "final_response": final_response,
            "extracted_answer": extracted,
            "model": model_name,
            "latency": round(latency, 2)
        })
        
        tqdm.write(f"  ✏️ #{qid:02d} [{category}/{difficulty}] "
                   f"耗时{latency:.1f}s → {extracted[:30]}{'...' if len(extracted) > 30 else ''}")
    
    print(f"\n✅ [{model_name}] 全部 {total} 题作答完成！")
    
    total_time = sum(r["latency"] for r in results)
    avg_time = total_time / total
    print(f"⏱️  总耗时: {total_time:.1f}s | 平均每题: {avg_time:.1f}s")
    
    return results

def save_results(results, filepath):
    """保存结果到 JSON 文件"""
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"💾 测试结果已保存至: {filepath}")

print("✅ 生成框架定义完成！")

# ==========================================================
# 🔍 案例观察：利用前两道题快速测试生成框架
# ==========================================================
print("\n👀 [案例观察] 开始进行极简测试...")
demo_system = "你是一个严谨的数学和逻辑推理助手。"
demo_results = generate_results(test_data[:2], chat_with_local, "Demo-Local", demo_system)

for i, r in enumerate(demo_results):
    print(f"\n👀 [案例观察] 第{i+1}题：", r['question'])
    print("🤖 - 思考：", r['thinking'])
    print("✏️ - 做答：", r['final_response'])
    print("💡 - 结果：", r['extracted_answer'])

### 3.3 本地模型作答

我们将设定一个基础的 System Prompt，直接观察未经特调下，Qwen2-0.5B 本地模型的表现。

In [ ]:
# TODO: 在此处自行设计适用于本地 Qwen2 模型的 system_prompt
local_system_prompt = "你是一个严谨的数学和逻辑推理助手。请仔细分析问题，逐步思考推理过程，最后明确给出最终答案。"

# 使用本地模型作答
local_results = generate_results(
    questions=test_data,
    chat_fn=chat_with_local,
    model_name="Qwen2-0.5B-Instruct",
    system_prompt=local_system_prompt
)

# 保存结果
save_results(local_results, "data/test1_answer_local.json")

### 3.4 API 推理模型作答

我们将使用同样的策略测试云端更加强大的 API 模型，观察其答题效果。

In [ ]:
# TODO: 在此处自行设计适用于云端大模型的 system_prompt
api_system_prompt = "你是一个精确的推理引擎。请仔细分析问题，给出清晰的推理过程，最后明确给出最终答案。"

# 使用 API 模型作答
api_results = generate_results(
    questions=test_data,
    chat_fn=chat_with_api,
    model_name=API_MODEL,
    system_prompt=api_system_prompt
)

# 保存结果
save_results(api_results, "data/test1_answer_api.json")


---
## 📊 挑战二：模型评估

现在两个模型都已经交卷了，该我们当「阅卷老师」了！

我们将使用在第 13 课学到的三种核心评估指标：

| 指标 | 全称 | 核心逻辑 | 适用场景 |
|------|------|----------|----------|
| **EM** | Exact Match | 答案完全一致才算对 | 选择题、数值计算 |
| **F1** | F1 Score | 字符级别的查准率 × 查全率 | 短文本问答 |
| **BLEU** | Bilingual Evaluation Understudy | N-gram + 短句惩罚 | 长文本生成 |

### 载入答案文件

In [ ]:
# 载入两份答案文件
with open("data/test1_answer_local.json", "r", encoding="utf-8") as f:
    local_results = json.load(f)

with open("data/test1_answer_api.json", "r", encoding="utf-8") as f:
    api_results = json.load(f)

print(f"📦 本地模型答案: {len(local_results)} 条")
print(f"📦 API 模型答案: {len(api_results)} 条")

# 预览对比
print("\n📝 答案对比预览（前3题）：")
print("-" * 80)
for i in range(min(3, len(local_results))):
    la = local_results[i]
    aa = api_results[i]
    print(f"题目 #{la['id']}: {la['question'][:40]}...")
    print(f"  标准答案: {la['expected_answer']}")
    print(f"  本地模型: {la['extracted_answer'][:50]}{'...' if len(la['extracted_answer']) > 50 else ''}")
    print(f"  API 模型: {aa['extracted_answer'][:50]}{'...' if len(aa['extracted_answer']) > 50 else ''}")
    print("-" * 80)

### 评估函数定义

所有评估函数遵循**统一的接口规范**：
```python
def eval_xxx(prediction: str, reference: str) -> float
```
- 输入：模型预测值 `prediction` 和标准答案 `reference`
- 输出：0.0 ~ 1.0 的分数

In [ ]:
def eval_exact_match(prediction, reference):
    """精确匹配 (Exact Match)
    
    将预测值和参考答案进行标准化后比较：
    - 去除前后空白字符
    - 统一转为小写
    - 去除标点符号
    
    Returns:
        1.0 如果完全匹配，否则 0.0
    """
    def normalize(text):
        text = str(text).strip().lower()
        # 去除常见标点
        text = re.sub(r'[，。！？、；：""''（）\[\]{}.,!?;:\'\"()\s]', '', text)
        return text
    
    return 1.0 if normalize(prediction) == normalize(reference) else 0.0


def eval_f1_score(prediction, reference):
    """字符级 F1 Score
    
    计算预测文本和参考文本的字符级别 F1 分数。
    F1 = 2 * Precision * Recall / (Precision + Recall)
    
    Returns:
        0.0 ~ 1.0 的 F1 分数
    """
    pred_chars = list(str(prediction).strip())
    ref_chars = list(str(reference).strip())
    
    if not pred_chars or not ref_chars:
        return 0.0
    
    # 计算重叠的字符数
    common_chars = []
    temp_ref = ref_chars.copy()
    for c in pred_chars:
        if c in temp_ref:
            common_chars.append(c)
            temp_ref.remove(c)
    
    common_len = len(common_chars)
    if common_len == 0:
        return 0.0
    
    precision = common_len / len(pred_chars)
    recall = common_len / len(ref_chars)
    return (2 * precision * recall) / (precision + recall)


def eval_bleu_score(prediction, reference):
    """BLEU Score
    
    计算预测文本和参考文本的 BLEU 分数。
    使用 1-gram 和 2-gram 权重各 0.5。
    
    Returns:
        0.0 ~ 1.0 的 BLEU 分数
    """
    pred_tokens = list(str(prediction).strip())
    ref_tokens = list(str(reference).strip())
    
    if not pred_tokens or not ref_tokens:
        return 0.0
    
    smoothie = SmoothingFunction().method1
    return sentence_bleu(
        [ref_tokens], pred_tokens,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smoothie
    )


# 验证评估函数
print("🧪 评估函数验证：")
print(f"  EM('42', '42')    = {eval_exact_match('42', '42')}")
print(f"  EM('四十二', '42') = {eval_exact_match('四十二', '42')}")
print(f"  F1('北京', '北京大学') = {eval_f1_score('北京', '北京大学'):.2f}")
print(f"  BLEU('北京', '北京大学') = {eval_bleu_score('北京', '北京大学'):.2f}")
print("\n✅ 评估函数定义完成！")

### 批量评估执行

我们将对两个模型的全部答案运行三种评估指标，生成详细的评估报告。

In [ ]:
def run_evaluation(answer_list, label="模型"):
    """对一组答案运行全部评估指标
    
    Args:
        answer_list: 答案列表 (包含 extracted_answer 和 expected_answer)
        label: 模型标签名
    
    Returns:
        eval_results: 包含每题评分的详细列表
    """
    eval_results = []
    
    for item in tqdm(answer_list, desc=f"评估 {label}", leave=False):
        pred = item["extracted_answer"]
        ref = item["expected_answer"]
        
        scores = {
            "id": item["id"],
            "category": item["category"],
            "difficulty": item["difficulty"],
            "question": item["question"][:30],
            "prediction": pred[:30],
            "reference": ref,
            "model": label,
            "latency": item["latency"],
            "em": eval_exact_match(pred, ref),
            "f1": eval_f1_score(pred, ref),
            "bleu": eval_bleu_score(pred, ref)
        }
        eval_results.append(scores)
    
    return eval_results


# 运行评估
local_eval = run_evaluation(local_results, label="Qwen2-0.5B")
api_eval = run_evaluation(api_results, label=API_MODEL)

# 合并为统一 DataFrame
df_eval = pd.DataFrame(local_eval + api_eval)

print(f"\n✅ 评估完成！共 {len(df_eval)} 条评估记录")

### 评测结果报告

#### 📋 总体评分

In [ ]:
# =============================================
# 总体评分汇总
# =============================================
print("🏆 总体评分汇总")
print("=" * 70)

summary = df_eval.groupby("model").agg({
    "em": "mean",
    "f1": "mean",
    "bleu": "mean",
    "latency": ["mean", "sum"]
}).round(4)

summary.columns = ["EM (准确率)", "F1 Score", "BLEU Score", "平均耗时(s)", "总耗时(s)"]
print(summary.to_string())

# =============================================
# 按类别分组评分
# =============================================
print("\n\n📊 按题目类型分组评分")
print("=" * 70)

category_summary = df_eval.groupby(["model", "category"]).agg({
    "em": "mean",
    "f1": "mean",
    "bleu": "mean"
}).round(4)
category_summary.columns = ["EM", "F1", "BLEU"]
print(category_summary.to_string())

# =============================================
# 按难度分组评分
# =============================================
print("\n\n📊 按难度级别分组评分")
print("=" * 70)

difficulty_summary = df_eval.groupby(["model", "difficulty"]).agg({
    "em": "mean",
    "f1": "mean",
    "bleu": "mean"
}).round(4)
difficulty_summary.columns = ["EM", "F1", "BLEU"]
print(difficulty_summary.to_string())

# =============================================
# 交叉分组：类型 × 难度
# =============================================
print("\n\n📊 类型 × 难度 交叉评分 (EM 准确率)")
print("=" * 70)

cross_summary = df_eval.pivot_table(
    values="em", 
    index=["model"], 
    columns=["category", "difficulty"], 
    aggfunc="mean"
).round(4)
print(cross_summary.to_string())

#### 📋 逐题详细对比

In [ ]:
# 逐题打印详细对比（只展示前 10 题）
print("\n📝 逐题详细对比（前 10 题）")
print("=" * 90)

header = f"{'ID':>3} | {'类型':^6} | {'难度':^6} | {'模型':^20} | {'EM':^4} | {'F1':^5} | {'耗时':^6} | 提取答案"
print(header)
print("-" * 90)

for qid in range(1, min(11, len(local_results) + 1)):
    le = local_eval[qid - 1]
    ae = api_eval[qid - 1]
    
    em_icon_l = "✅" if le["em"] == 1.0 else "❌"
    em_icon_a = "✅" if ae["em"] == 1.0 else "❌"
    
    print(f"#{qid:02d} | {le['category']:^6} | {le['difficulty']:^6} | "
          f"{'Qwen2-0.5B':^20} | {em_icon_l:^4} | {le['f1']:.2f} | {le['latency']:5.1f}s | {le['prediction']}")
    print(f"{'':>3} | {'':^6} | {'':^6} | "
          f"{API_MODEL[:20]:^20} | {em_icon_a:^4} | {ae['f1']:.2f} | {ae['latency']:5.1f}s | {ae['prediction']}")
    print(f"{'':>3} | {'':^6} | {'':^6} | "
          f"{'[标准答案]':^20} | {'':^4} | {'':^5} | {'':^6} | {le['reference']}")
    print("-" * 90)

---
## 📈 结果可视化

### EM 准确率对比（按类型 × 难度）

In [ ]:
# 准备数据
models = df_eval["model"].unique()
categories = ["math", "logic"]
difficulties = ["easy", "medium", "hard"]
group_labels = [f"{cat}\n{diff}" for cat in categories for diff in difficulties]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---- 图1: EM 准确率分组对比 ----
ax1 = axes[0]
x = np.arange(len(group_labels))
width = 0.35

for i, m in enumerate(models):
    em_scores = []
    for cat in categories:
        for diff in difficulties:
            subset = df_eval[(df_eval["model"] == m) & 
                            (df_eval["category"] == cat) & 
                            (df_eval["difficulty"] == diff)]
            em_scores.append(subset["em"].mean() if len(subset) > 0 else 0)
    
    offset = (i - 0.5) * width
    bars = ax1.bar(x + offset, em_scores, width, label=m, alpha=0.85)
    # 在柱子上标注数值
    for bar, score in zip(bars, em_scores):
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f"{score:.0%}", ha='center', va='bottom', fontsize=8)

ax1.set_xlabel("题目分组")
ax1.set_ylabel("EM 准确率")
ax1.set_title("Exact Match 准确率对比")
ax1.set_xticks(x)
ax1.set_xticklabels(group_labels)
ax1.set_ylim(0, 1.15)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# ---- 图2: 平均耗时对比 ----
ax2 = axes[1]

for i, m in enumerate(models):
    latencies = []
    for cat in categories:
        for diff in difficulties:
            subset = df_eval[(df_eval["model"] == m) & 
                            (df_eval["category"] == cat) & 
                            (df_eval["difficulty"] == diff)]
            latencies.append(subset["latency"].mean() if len(subset) > 0 else 0)
    
    offset = (i - 0.5) * width
    bars = ax2.bar(x + offset, latencies, width, label=m, alpha=0.85)
    for bar, lat in zip(bars, latencies):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                f"{lat:.1f}s", ha='center', va='bottom', fontsize=8)

ax2.set_xlabel("题目分组")
ax2.set_ylabel("平均耗时 (秒)")
ax2.set_title("推理耗时对比")
ax2.set_xticks(x)
ax2.set_xticklabels(group_labels)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/reasoning_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 图表已保存至 outputs/reasoning_comparison.png")

### 综合能力雷达图

In [ ]:
# 构建雷达图数据
dimensions = ["math_em", "logic_em", "avg_f1", "avg_bleu", "speed"]
dim_labels = ["数学准确率", "逻辑准确率", "F1 得分", "BLEU 得分", "响应速度"]

radar_data = {}
for m in models:
    m_data = df_eval[df_eval["model"] == m]
    math_em = m_data[m_data["category"] == "math"]["em"].mean()
    logic_em = m_data[m_data["category"] == "logic"]["em"].mean()
    avg_f1 = m_data["f1"].mean()
    avg_bleu = m_data["bleu"].mean()
    # 速度用反比映射：耗时越短得分越高
    avg_latency = m_data["latency"].mean()
    speed_score = min(1.0, 5.0 / (avg_latency + 0.1))  # 5秒以内算满分
    
    radar_data[m] = [math_em, logic_em, avg_f1, avg_bleu, speed_score]

# 绘制雷达图
angles = np.linspace(0, 2 * np.pi, len(dimensions), endpoint=False).tolist()
angles += angles[:1]  # 闭合

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colors = ['#2ecc71', '#e74c3c']
for idx, (m, values) in enumerate(radar_data.items()):
    values_closed = values + values[:1]
    ax.plot(angles, values_closed, 'o-', linewidth=2, label=m, color=colors[idx % 2])
    ax.fill(angles, values_closed, alpha=0.15, color=colors[idx % 2])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dim_labels, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title("模型综合能力雷达图", fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/reasoning_radar.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 雷达图已保存至 outputs/reasoning_radar.png")

---
## 📝 实验总结与思考（建议在实验报告里谈一谈）

### 实验收获

通过本次综合实验，我们完整实践了大模型推理能力评测的全流程：

1. **模型加载与调用**：掌握了本地模型推理和远端 API 调用两种主流的模型使用方式。
2. **Prompt 工程**：通过设计 System Prompt，尝试引导模型输出更高质量的推理过程。
3. **答案提取**：学习了从非结构化的模型输出中提取关键信息的正则表达式技巧。
4. **科学评估**：使用 EM、F1、BLEU 三种指标从不同维度量化评价模型表现。
5. **数据可视化**：通过柱状图和雷达图直观呈现对比结果。

### 💡 思考题

1. 你设计的 System Prompt 对模型的准确率有多大影响？尝试不同的 Prompt 会怎样？
2. 在数学题和逻辑题上，EM/F1/BLEU能反映模型的真实水平吗？为什么？
3. 如果要让小模型在推理任务上有所提升，你觉得可以从哪些方向努力？
4. 【可选】如果尝试了Self-verification或者step-verification，可以谈谈你的感想。